# Baselines on CAGE-2: R-PPO, DR-RPPO, VariBAD, and MMD
Every baseline acts on the same observation interface, blue's tracked state, its latest signal, and its previous action $[z^S_k, y_k, a^1_{k-1}]$, through a recurrent policy, and every baseline trains for the same number of environment steps as our method (`BUDGET_STEPS`). Red's head start is played by the training opponent. The baselines learn from the physical payoff `Reward2`, the evaluation payoff.

- **R-PPO**: recurrent PPO against the trained red.
- **DR-RPPO**: recurrent PPO against a red drawn per episode from a pool (uniformly random, the scripted `logical_red_policy`, and the trained red).
- **VariBAD**: Bayes-adaptive meta-RL, conditioning the policy on an inferred task belief (mean and standard deviation of a recurrent variational encoder trained to predict the next signal).
- **MMD**: magnetic mirror descent in self-play, regularizing each player toward a slowly moving magnet copy of itself; it respects the game but never represents the context.

`RecurrentAC` and `VariBADEncoder` are defined in `Initialize.ipynb`. Checkpoints are saved in `compared/`, each as a dictionary holding the network weights, the configuration, and the per-update training returns.

In [ ]:
%run Initialize.ipynb

## Shared settings

In [ ]:
from copy import deepcopy
from torch.distributions import kl_divergence

# Matched interaction budget: main.ipynb trains our method for 10,000 episodes x 50 rounds
BUDGET_STEPS     = 10000 * max_steps
TOTAL_EPISODES   = BUDGET_STEPS // max_steps     # 10,000 episodes for every baseline
EPISODES_PER_UPD = 10
GAMMA            = 0.99
GAE_LAMBDA       = 0.95
PPO_EPOCHS       = 4
CLIP_EPS         = 0.2
ENT_COEF         = 0.01
VF_COEF          = 0.5
LR               = 3e-4
VARIBAD_KL_W     = 0.05
MMD_COEF         = 0.05
MAGNET_TAU       = 0.1
CKPT_EVERY       = 200
SAVE_DIR         = "compared"
os.makedirs(SAVE_DIR, exist_ok=True)
BUDGET = {}                                      # environment steps used by each baseline

trained_red = red_policy(Load(AttackerActor().to(device), "player2.pth"))


def sample_config():
    # Context, then a simulator reset with red's reconnaissance, in the same order as main.ipynb
    Theta, Type_Theta, w, p_exploit, h = Sample_Context()
    blue_state, blue_knowledge, red_state, red_knowledge, real_state = Reset_Episode()
    return Theta, Type_Theta, w, p_exploit, h, blue_state, red_state, red_knowledge, real_state

## Red pool

In [ ]:
class OpponentTrainedRed:
    def reset(self, Theta, Type_Theta, w, p_exploit):
        self.Type_Theta = Type_Theta
        self.p_exploit = p_exploit
    def act(self, red_state):
        return trained_red(red_state, self.Type_Theta)

class OpponentRandomRed:
    def reset(self, Theta, Type_Theta, w, p_exploit):
        self.p_exploit = p_exploit
    def act(self, red_state):
        return int(np.random.randint(n_u))

class OpponentLogicalRed:
    def reset(self, Theta, Type_Theta, w, p_exploit):
        self.Type_Theta = Type_Theta
        self.p_exploit = p_exploit
    def act(self, red_state):
        return logical_red_policy(red_state, self.Type_Theta)

POOL_FACTORIES = [OpponentRandomRed, OpponentLogicalRed, OpponentTrainedRed]

## Rollouts and PPO

In [ ]:
def gae(rews, vals):
    T = len(rews)
    adv = np.zeros(T, dtype=np.float32); last = 0.0
    for t in reversed(range(T)):
        nv = vals[t + 1] if t + 1 < T else 0.0
        delta = rews[t] + GAMMA * nv - vals[t]
        last = delta + GAMMA * GAE_LAMBDA * last
        adv[t] = last
    return adv, adv + np.asarray(vals, dtype=np.float32)

def rollout_episode(policy, opponent, hook=None):
    Theta, Type_Theta, w, p_exploit, h, blue_state, red_state, red_knowledge, real_state = sample_config()
    opponent.reset(Theta, Type_Theta, w, p_exploit)
    for _ in range(h):
        u = opponent.act(red_state)
        red_action_name, cyborg_action = Red_Action_Gen(u, red_knowledge, p_exploit)
        res_red = cyborg.step(agent='Red', action=cyborg_action)
        real_state = System_Real_State(cyborg.environment_controller.state)
        red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
        red_state = red_state_vector(res_red.observation, red_knowledge, red_state, red_action_name)
    y_cur = np.zeros(real_state_dim, dtype=np.float32)
    a_prev = np.zeros(n_a, dtype=np.float32)
    gru_h = None
    if hook is not None:
        hook.reset()
    obs_l, ex_l, a_l, lp_l, v_l, r_l, y_l, av_l = [], [], [], [], [], [], [], []
    for t in range(max_steps):
        obs = np.concatenate([blue_state, y_cur, a_prev]).astype(np.float32)
        obs_t = torch.tensor(obs, device=device).view(1, 1, OBS_DIM)
        ex_np = hook.extra(obs) if hook is not None else None
        ex_t = None if ex_np is None else torch.tensor(
            np.asarray(ex_np, dtype=np.float32), device=device).view(1, 1, -1)
        with torch.no_grad():
            logits, val, gru_h = policy(obs_t, ex_t, gru_h)
        dist = Categorical(logits=logits.view(-1))
        a = int(dist.sample())
        lp = float(dist.log_prob(torch.tensor(a, device=device)))
        u = opponent.act(red_state)
        red_action_name, cyborg_action = Red_Action_Gen(u, red_knowledge, p_exploit)
        res_red = cyborg.step(agent='Red', action=cyborg_action)
        real_state = System_Real_State(cyborg.environment_controller.state)
        red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
        next_red_state = red_state_vector(res_red.observation, red_knowledge, red_state, red_action_name)
        blue_action_name, cyborg_action, host = Blue_Action_Gen(a)
        res = cyborg.step(agent='Blue', action=cyborg_action)
        y = extract_y(res.observation, blue_action_name)
        next_blue_state = blue_state_vector(y, blue_state, blue_action_name, t + 1)
        next_real_state = System_Real_State(cyborg.environment_controller.state)
        r = float(Reward2(next_real_state, w))
        y_clipped = np.maximum(y, 0).astype(np.float32)
        a_onehot = np.zeros(n_a, dtype=np.float32); a_onehot[a] = 1.0
        if hook is not None:
            hook.observe(y_cur, a_onehot, y_clipped)
        obs_l.append(obs); a_l.append(a); lp_l.append(lp); v_l.append(val.item())
        r_l.append(r); y_l.append(y_clipped); av_l.append(a_onehot)
        if ex_np is not None:
            ex_l.append(np.asarray(ex_np, dtype=np.float32))
        blue_state = next_blue_state
        red_state = next_red_state
        real_state = next_real_state
        y_cur = y_clipped
        a_prev = a_onehot
    adv, ret = gae(np.array(r_l, dtype=np.float32), np.array(v_l, dtype=np.float32))
    ep = {'obs': np.stack(obs_l), 'acts': np.array(a_l), 'logp': np.array(lp_l, dtype=np.float32),
          'adv': adv, 'ret': ret, 'rews': np.array(r_l, dtype=np.float32),
          'ys': np.stack(y_l), 'avecs': np.stack(av_l)}
    if ex_l:
        ep['extras'] = np.stack(ex_l)
    return ep

def stack_batch(eps, keys):
    out = []
    for k in keys:
        dt = torch.long if k == 'acts' else torch.float32
        out.append(torch.tensor(np.stack([e[k] for e in eps]), dtype=dt, device=device))
    return out

def ppo_update(policy, opt, eps, extras=None, extra_params=None):
    obs, acts, oldlp, adv, ret = stack_batch(eps, ['obs', 'acts', 'logp', 'adv', 'ret'])
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    params = list(policy.parameters()) + (list(extra_params) if extra_params else [])
    for _ in range(PPO_EPOCHS):
        logits, values, _ = policy(obs, extras)
        dist = Categorical(logits=logits)
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg = -torch.min(ratio * adv,
                        torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
        vl = F.mse_loss(values, ret)
        ent = dist.entropy().mean()
        loss = pg + VF_COEF * vl - ENT_COEF * ent
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
    return float(pg.item()), float(vl.item())

In [ ]:
def train_recurrent_ppo(save_name, opponent_sampler, seed):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    policy = RecurrentAC(extra_dim=0).to(device)
    opt = optim.AdamW(policy.parameters(), lr=LR)
    n_upd = TOTAL_EPISODES // EPISODES_PER_UPD
    ret_hist = []
    for upd in tqdm(range(n_upd), desc=save_name):
        eps = [rollout_episode(policy, opponent_sampler())
               for _ in range(EPISODES_PER_UPD)]
        ppo_update(policy, opt, eps)
        ret_hist.append(float(np.mean([e['rews'].sum() for e in eps])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'policy': policy.state_dict(),
                        'config': {'RNN_H': RNN_H, 'extra_dim': 0},
                        'return_per_episode': ret_hist},
                       os.path.join(SAVE_DIR, save_name + '.pth'))
        if (upd + 1) % 25 == 0:
            print(f"{save_name} upd {upd+1}/{n_upd}  return/episode (last 25 upd): "
                  f"{np.mean(ret_hist[-25:]):.3f}")
    BUDGET[save_name] = n_upd * EPISODES_PER_UPD * max_steps
    return policy, ret_hist

## R-PPO: recurrent PPO against the trained red

In [ ]:
RPPO_policy, RPPO_hist = train_recurrent_ppo('RPPO', lambda: OpponentTrainedRed(), seed=101)

## DR-RPPO: recurrent PPO against a red drawn per episode from the pool

In [ ]:
DR_policy, DR_hist = train_recurrent_ppo('DR_RPPO', lambda: random.choice(POOL_FACTORIES)(), seed=202)

## VariBAD: policy conditioned on a variational task belief

In [ ]:
class VariBADDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(LATENT_DIM + real_state_dim + n_a, 128), nn.ReLU(),
                                 nn.Linear(128, real_state_dim))
    def forward(self, z, y, a):
        return self.net(torch.cat([z, y, a], dim=-1))

class VariBADHook:
    def __init__(self, enc):
        self.enc = enc
    def reset(self):
        self.h = None
    def extra(self, obs):
        with torch.no_grad():
            mu, logvar, self.h = self.enc(
                torch.tensor(obs, dtype=torch.float32, device=device).view(1, 1, -1), self.h)
        mu = mu.view(-1).cpu().numpy()
        std = np.exp(0.5 * logvar.view(-1).cpu().numpy())
        return np.concatenate([mu, std])
    def observe(self, y, a, y_next):
        pass

def train_varibad(seed=303):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    enc = VariBADEncoder().to(device)
    dec = VariBADDecoder().to(device)
    policy = RecurrentAC(extra_dim=2 * LATENT_DIM).to(device)
    enc_opt = optim.AdamW(list(enc.parameters()) + list(dec.parameters()), lr=LR)
    pol_opt = optim.AdamW(policy.parameters(), lr=LR)
    hook = VariBADHook(enc)
    n_upd = TOTAL_EPISODES // EPISODES_PER_UPD
    ret_hist = []
    for upd in tqdm(range(n_upd), desc='VARIBAD'):
        eps = [rollout_episode(policy, OpponentTrainedRed(), hook)
               for _ in range(EPISODES_PER_UPD)]
        obs, ys, avecs = stack_batch(eps, ['obs', 'ys', 'avecs'])
        mu, logvar, _ = enc(obs)
        std = (0.5 * logvar).exp()
        z = mu + std * torch.randn_like(std)
        recon = dec(z, obs[..., blue_state_dim:blue_state_dim + real_state_dim], avecs)
        kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(-1).mean()
        elbo = F.mse_loss(recon, ys) + VARIBAD_KL_W * kl
        enc_opt.zero_grad(); elbo.backward()
        torch.nn.utils.clip_grad_norm_(list(enc.parameters()) + list(dec.parameters()), 1.0)
        enc_opt.step()
        extras = stack_batch(eps, ['extras'])[0]
        ppo_update(policy, pol_opt, eps, extras=extras)
        ret_hist.append(float(np.mean([e['rews'].sum() for e in eps])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'policy': policy.state_dict(),
                        'encoder': enc.state_dict(),
                        'decoder': dec.state_dict(),
                        'config': {'RNN_H': RNN_H, 'LATENT_DIM': LATENT_DIM,
                                   'extra_dim': 2 * LATENT_DIM},
                        'return_per_episode': ret_hist},
                       os.path.join(SAVE_DIR, 'VARIBAD.pth'))
        if (upd + 1) % 25 == 0:
            print(f"VARIBAD upd {upd+1}/{n_upd}  return/episode (last 25 upd): "
                  f"{np.mean(ret_hist[-25:]):.3f}  elbo {elbo.item():.4f}")
    BUDGET['VARIBAD'] = n_upd * EPISODES_PER_UPD * max_steps
    return policy, enc, dec, ret_hist

VB_policy, VB_enc, VB_dec, VB_hist = train_varibad()

## MMD: magnetic mirror descent in self-play

In [ ]:
class P2AC(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(red_state_dim + 2, 256), nn.ReLU(),
                                   nn.Linear(256, 256), nn.ReLU())
        self.pi = nn.Linear(256, n_u)
        self.v = nn.Linear(256, 1)
    def forward(self, inp):
        h = self.trunk(inp)
        return self.pi(h), self.v(h).squeeze(-1)

def rollout_selfplay(p1, p2):
    Theta, Type_Theta, w, p_exploit, h, blue_state, red_state, red_knowledge, real_state = sample_config()
    for _ in range(h):
        inp2 = np.concatenate([red_state, Type_Theta[:2]]).astype(np.float32)
        with torch.no_grad():
            logits2, _ = p2(torch.tensor(inp2, device=device).unsqueeze(0))
        u = int(Categorical(logits=logits2.view(-1)).sample())
        red_action_name, cyborg_action = Red_Action_Gen(u, red_knowledge, p_exploit)
        res_red = cyborg.step(agent='Red', action=cyborg_action)
        real_state = System_Real_State(cyborg.environment_controller.state)
        red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
        red_state = red_state_vector(res_red.observation, red_knowledge, red_state, red_action_name)
    y_cur = np.zeros(real_state_dim, dtype=np.float32)
    a_prev = np.zeros(n_a, dtype=np.float32)
    gru_h = None
    o1, a1, l1, v1, r1 = [], [], [], [], []
    i2, u2, l2, v2 = [], [], [], []
    for t in range(max_steps):
        obs = np.concatenate([blue_state, y_cur, a_prev]).astype(np.float32)
        with torch.no_grad():
            logits, val, gru_h = p1(torch.tensor(obs, device=device).view(1, 1, OBS_DIM), None, gru_h)
        d1 = Categorical(logits=logits.view(-1))
        a = int(d1.sample())
        inp2 = np.concatenate([red_state, Type_Theta[:2]]).astype(np.float32)
        with torch.no_grad():
            logits2, val2 = p2(torch.tensor(inp2, device=device).unsqueeze(0))
        d2 = Categorical(logits=logits2.view(-1))
        u_base = int(d2.sample())
        red_action_name, cyborg_action = Red_Action_Gen(u_base, red_knowledge, p_exploit)
        res_red = cyborg.step(agent='Red', action=cyborg_action)
        real_state = System_Real_State(cyborg.environment_controller.state)
        red_knowledge = red_knowledge_dic(res_red.observation, red_knowledge)
        next_red_state = red_state_vector(res_red.observation, red_knowledge, red_state, red_action_name)
        blue_action_name, cyborg_action, host = Blue_Action_Gen(a)
        res = cyborg.step(agent='Blue', action=cyborg_action)
        y = extract_y(res.observation, blue_action_name)
        next_blue_state = blue_state_vector(y, blue_state, blue_action_name, t + 1)
        next_real_state = System_Real_State(cyborg.environment_controller.state)
        r = float(Reward2(next_real_state, w))
        y_clipped = np.maximum(y, 0).astype(np.float32)
        a_onehot = np.zeros(n_a, dtype=np.float32); a_onehot[a] = 1.0
        o1.append(obs); a1.append(a)
        l1.append(float(d1.log_prob(torch.tensor(a, device=device))))
        v1.append(val.item()); r1.append(r)
        i2.append(inp2); u2.append(u_base)
        l2.append(float(d2.log_prob(torch.tensor(u_base, device=device))))
        v2.append(val2.item())
        blue_state = next_blue_state
        red_state = next_red_state
        real_state = next_real_state
        y_cur = y_clipped
        a_prev = a_onehot
    r1 = np.array(r1, dtype=np.float32)
    adv1, ret1 = gae(r1, np.array(v1, dtype=np.float32))
    adv2, ret2 = gae(-r1, np.array(v2, dtype=np.float32))
    ep1 = {'obs': np.stack(o1), 'acts': np.array(a1),
           'logp': np.array(l1, dtype=np.float32), 'adv': adv1, 'ret': ret1, 'rews': r1}
    ep2 = {'inp': np.stack(i2), 'acts': np.array(u2),
           'logp': np.array(l2, dtype=np.float32), 'adv': adv2, 'ret': ret2}
    return ep1, ep2

def mmd_update_p1(p1, mag1, opt, eps):
    obs, acts, oldlp, adv, ret = stack_batch(eps, ['obs', 'acts', 'logp', 'adv', 'ret'])
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    for _ in range(PPO_EPOCHS):
        logits, values, _ = p1(obs)
        dist = Categorical(logits=logits)
        with torch.no_grad():
            mag_logits, _, _ = mag1(obs)
        kl_mag = kl_divergence(dist, Categorical(logits=mag_logits)).mean()
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg = -torch.min(ratio * adv,
                        torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
        vl = F.mse_loss(values, ret)
        loss = pg + VF_COEF * vl - ENT_COEF * dist.entropy().mean() + MMD_COEF * kl_mag
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(p1.parameters(), 1.0)
        opt.step()

def mmd_update_p2(p2, mag2, opt, eps2):
    inp = torch.tensor(np.stack([e['inp'] for e in eps2]).reshape(-1, red_state_dim + 2),
                       dtype=torch.float32, device=device)
    acts = torch.tensor(np.stack([e['acts'] for e in eps2]).reshape(-1),
                        dtype=torch.long, device=device)
    oldlp = torch.tensor(np.stack([e['logp'] for e in eps2]).reshape(-1),
                         dtype=torch.float32, device=device)
    adv = torch.tensor(np.stack([e['adv'] for e in eps2]).reshape(-1),
                       dtype=torch.float32, device=device)
    ret = torch.tensor(np.stack([e['ret'] for e in eps2]).reshape(-1),
                       dtype=torch.float32, device=device)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    for _ in range(PPO_EPOCHS):
        logits, values = p2(inp)
        dist = Categorical(logits=logits)
        with torch.no_grad():
            mag_logits, _ = mag2(inp)
        kl_mag = kl_divergence(dist, Categorical(logits=mag_logits)).mean()
        lp = dist.log_prob(acts)
        ratio = (lp - oldlp).exp()
        pg = -torch.min(ratio * adv,
                        torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv).mean()
        vl = F.mse_loss(values, ret)
        loss = pg + VF_COEF * vl - ENT_COEF * dist.entropy().mean() + MMD_COEF * kl_mag
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(p2.parameters(), 1.0)
        opt.step()

def ema_(magnet, net):
    with torch.no_grad():
        for m, c in zip(magnet.parameters(), net.parameters()):
            m.mul_(1.0 - MAGNET_TAU).add_(MAGNET_TAU * c)

def train_mmd(seed=505):
    np.random.seed(seed); torch.manual_seed(seed); random.seed(seed)
    p1 = RecurrentAC(extra_dim=0).to(device)
    p2 = P2AC().to(device)
    mag1 = deepcopy(p1); mag2 = deepcopy(p2)
    mag1.gru.flatten_parameters()
    for m in list(mag1.parameters()) + list(mag2.parameters()):
        m.requires_grad_(False)
    opt1 = optim.AdamW(p1.parameters(), lr=LR)
    opt2 = optim.AdamW(p2.parameters(), lr=LR)
    n_upd = TOTAL_EPISODES // EPISODES_PER_UPD
    ret_hist = []
    for upd in tqdm(range(n_upd), desc='MMD'):
        pairs = [rollout_selfplay(p1, p2)
                 for _ in range(EPISODES_PER_UPD)]
        eps1 = [a for a, b in pairs]; eps2 = [b for a, b in pairs]
        mmd_update_p1(p1, mag1, opt1, eps1)
        mmd_update_p2(p2, mag2, opt2, eps2)
        ema_(mag1, p1); ema_(mag2, p2)
        ret_hist.append(float(np.mean([e['rews'].sum() for e in eps1])))
        if (upd + 1) % CKPT_EVERY == 0 or upd == n_upd - 1:
            torch.save({'p1': p1.state_dict(), 'p2': p2.state_dict(),
                        'p1_magnet': mag1.state_dict(), 'p2_magnet': mag2.state_dict(),
                        'config': {'RNN_H': RNN_H, 'extra_dim': 0,
                                   'MMD_COEF': MMD_COEF, 'MAGNET_TAU': MAGNET_TAU},
                        'return_per_episode': ret_hist},
                       os.path.join(SAVE_DIR, 'MMD.pth'))
        if (upd + 1) % 25 == 0:
            print(f"MMD upd {upd+1}/{n_upd}  P1 return/episode (last 25 upd): "
                  f"{np.mean(ret_hist[-25:]):.3f}")
    BUDGET['MMD'] = n_upd * EPISODES_PER_UPD * max_steps
    return p1, p2, ret_hist

MMD_p1, MMD_p2, MMD_hist = train_mmd()

## Interaction budgets

In [ ]:
print(f"{'method':8s} {'env steps':>10s}")
print(f"{'Ours':8s} {BUDGET_STEPS:10,d}   (main.ipynb)")
for k, v in BUDGET.items():
    print(f"{k:8s} {v:10,d}")